(User_MolecularSystem_Overview)=
# A molecular system in MolSysMT

MolSysViewer builds on top of [MolSysMT](https://www.uibcdf.org/molsysmt): MolSysMT defines what a **molecular system** is, how it can be represented in different forms, and how you can *query it* (for example, selecting atoms or inspecting topology).

If you are new to MolSysMT, this page is meant to give you a friendly mental model. You do not need to memorize anything—just aim to recognize the main pieces.

In practice, this matters because:

- MolSysViewer keeps the molecular system you are visualizing inside the viewer object.
- Atom selection in MolSysViewer is powered by MolSysMT selection.
- Loading molecular systems from files, third-party python libraries, or web databases, into MolSysViewer relies on MolSysMT conversions under the hood.


## The two key ideas: *forms* and *attributes*

MolSysMT is designed around two simple ideas:

1. A molecular system can exist in different **forms** (different Python objects / file formats) that represent the same underlying system.
2. Each form exposes a set of **attributes** (for example: atom names, group names, coordinates, box vectors, time).

MolSysMT provides a consistent toolbox (functions like `select`, `info`, `get`, and `convert`) that works across those forms.

MolSysViewer takes advantage of this: you can load different inputs, and MolSysMT will convert them to a viewer-friendly representation.

MolSysMT uses a shared vocabulary of attribute names (for example `atom_name`, `group_name`, `chain_id`, `coordinates`, `box`, …).
You will encounter these names in both selection strings and attribute queries.

Here is a quick reference table:

```{include} ../_snippets/attributes_table.md
```


In [1]:
import molsysviewer as viewer
import molsysmt as msm

## A concrete example: a demo molecular system

Let’s load a small demo trajectory (pentalanine). This returns a `MolSysView`, MolSysViewer’s main viewer object.

You can run this notebook cell-by-cell. If you are following the web documentation, treat these as examples you can reproduce locally.


In [2]:
view = viewer.demo["pentalanine"]
view.show()

In [3]:
from molsysviewer.thirds.jupyter import load_html_in_notebook
load_html_in_notebook('../../../_static/views/demo_pentalanine.html')

The molecular system being visualized is available as `view.molsys`.

This is a **read-only** property (there is no setter): you can inspect it freely, but if you mutate it in-place you are changing the very system the viewer is representing.

If you need to modify the loaded system, prefer MolSysViewer’s live methods (for example, `view.set(...)`, `view.remove(...)`, `view.add(...)`, or `view.append_structures(...)`) instead of modifying `view.molsys` directly.

In [4]:
type(view.molsys)

molsysmt.native.molsys.MolSys

## Topology vs structures

MolSysMT’s native molecular system (`molsysmt.MolSys`) separates the system into two big parts:

- **Topology**: “what is connected to what” and how atoms are organized (atoms, groups, components, chains, molecules, entities, bonds…).
- **Structures**: “where things are” (coordinates), and other per-frame information (box, time, structure identifiers…).

A trajectory is a single topology with many structures (frames). That is exactly why this split is so convenient.

MolSysMT uses the word *element* to refer to these hierarchical levels. Here is a quick reference table:

```{include} ../_snippets/elements_table.md
```


In [5]:
topology = view.molsys.topology
structures = view.molsys.structures

If you are curious, you can inspect the topology tables directly. For example, atoms live in a Pandas table.


In [6]:
topology.atoms.head()

,atom_id,atom_name,atom_type,group_index,component_index,chain_index
0,0,H1,H,0,0,0
1,1,CH3,C,0,0,0
2,2,H2,H,0,0,0
3,3,H3,H,0,0,0
4,4,C,C,0,0,0


And structures store coordinate arrays and optional per-frame attributes.


In [7]:
structures.coordinates.shape

(200, 62, 3)

You will find further details about `molsysmt.MolSys`, and about its inner objects for topology and structures, in the MolSysMT documentation.

MolSysViewer can be used without knowing MolSysMT. Still, learning a few MolSysMT basics is strongly recommended if you want to get the most out of MolSysViewer.

## Querying the system: `info`, `select`, and `get`

MolSysViewer exposes `view.info(...)`, `view.select(...)`, and `view.get(...)` as convenience wrappers around MolSysMT’s
`molsysmt.info(...)`, `molsysmt.select(...)`, and `molsysmt.get(...)`.

Because the loaded molecular system lives in `view.molsys` as a MolSysMT object, you can combine MolSysMT’s querying power
with Mol\*’s rendering power in a single workflow.

In a notebook, `info` gives you a friendly summary table.

In [8]:
view.info(element="system")

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,62,7,1,1,1,1,1,200


Selections are also handled by MolSysMT.

As a small example, let’s select the alpha-carbon atoms (`CA`) and then display their information.

If you want to go deeper on selection language and examples, see MolSysMT’s documentation.


In [9]:
ca_atom_indices = view.select(selection="atom_name == 'CA'")
ca_atom_indices

[8, 18, 28, 38, 48]

In [10]:
view.info(element="atom", selection=ca_atom_indices)

index,id,name,type,group index,group id,group name,group type,component index,chain index,molecule index,molecule type,entity index,entity name
8,8,CA,C,1,2,ALA,amino acid,0,0,0,peptide,0,peptide 0
18,18,CA,C,2,3,ALA,amino acid,0,0,0,peptide,0,peptide 0
28,28,CA,C,3,4,ALA,amino acid,0,0,0,peptide,0,peptide 0
38,38,CA,C,4,5,ALA,amino acid,0,0,0,peptide,0,peptide 0
48,48,CA,C,5,6,ALA,amino acid,0,0,0,peptide,0,peptide 0


If you need specific values of attributes, use `get`:

In [11]:
view.get(element="system", n_atoms=True, n_structures=True)

[62, 200]

In [12]:
view.get(element="atom", selection=ca_atom_indices[:5], atom_name=True, group_name=True)

[['CA', 'CA', 'CA', 'CA', 'CA'], ['ALA', 'ALA', 'ALA', 'ALA', 'ALA']]

More on these three functions will be found in the following sections of this documentation.

## Conversion is the bridge to the viewer

The browser viewer cannot consume arbitrary Python objects. MolSysViewer needs a lightweight, viewer-friendly payload.

This is where MolSysMT’s `convert` comes in: it can turn a molecular system into a minimal JSON-like representation named `molsysmt.ViewerJSON`.

MolSysViewer uses this idea internally in the loading pipeline (`new_view` / `load`). You normally do not have to call `convert` yourself—but seeing it once can help you understand what is happening behind the scenes.


In [13]:
viewer_json = msm.convert(view.molsys, to_form="molsysmt.ViewerJSON")
type(viewer_json)

molsysmt.native.viewer_json.ViewerJSON

In [14]:
payload = viewer_json.to_dict(copy=False)
list(payload.keys())

['version', 'atoms', 'bonds', 'structures']

Try running the following code in your own notebook:

```python
import molsysviewer as viewer

view = viewer.new_view(
    "181L",
    selection='molecule_type not in ["water", "ion"]',
)
view.show()
```

You should see the molecular system from the Protein Data Bank entry **181L**, with crystallographic waters and ions removed.

This is a good example of what “conversion” means in practice:

- MolSysViewer does not implement a special-case “PDB id loader”.
- MolSysMT is the layer that understands inputs like a PDB id string, applies the selection, and converts the result into a viewer-friendly payload.
- Mol\* is the rendering engine that draws the scene in the browser.

## Takeaway

If you only keep one idea from this page, let it be this:

- **MolSysMT defines and manipulates the molecular system** (topology + structures, selection, inspection, conversions).
- **Mol\* renders the 3D scene in the browser.**
- **MolSysViewer focuses on interactive visualization for analysis and exploration**, and relies on MolSysMT to understand the system and on Mol\* to create the visualization.

In the next User Guide units, we will build on this model to explain selection, regions, layers, representations, and overlays.